# RecON — Google Colab 运行环境

**使用前请确认：**
1. 已将本项目推送到自己的 GitHub 仓库（见本文档步骤0）
2. 已将 `data/` 目录上传至 Google Drive
3. 运行时类型已切换为 **GPU**（菜单 → 运行时 → 更改运行时类型 → T4 GPU）

## 第一步：挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 确认 Drive 挂载成功
print('Drive 已挂载:', os.path.exists('/content/drive/MyDrive'))

## 第二步：克隆代码仓库

将下方 `GITHUB_REPO` 替换为你自己的仓库地址（`https://github.com/<你的用户名>/RecON.git`）。

In [ ]:
GITHUB_REPO = 'https://github.com/<YOUR_USERNAME>/RecON.git'  # ← 修改此处
BRANCH = 'main'  # 如使用其他分支请修改
PROJECT_DIR = '/content/RecON'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO} {PROJECT_DIR}
else:
    print('目录已存在，执行 git pull 更新...')
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print('当前目录:', os.getcwd())

## 第三步：安装依赖

Colab 已预装 PyTorch，只需补充项目额外依赖。

In [ ]:
# 确认 PyTorch 版本及 CUDA 可用性
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# 安装额外依赖（timm、pyvista、h5py；opencv/scipy/numpy 已预装）
!pip install timm pyvista h5py --quiet

# 验证所有依赖可正常导入
import importlib
for pkg in ['cv2', 'numpy', 'scipy', 'timm', 'h5py', 'pyvista']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError as e:
        print(f'  ✗ {pkg}: {e}')

## 第四步：挂载数据

**前置操作（在本地执行一次）：**  
将本地 `data/` 目录（含 `frames_transfs/` 和 `calib_matrix.csv`）上传到 Google Drive，
建议路径：`我的云端硬盘/RecON_data/`

上传后 Drive 内的结构应为：
```
我的云端硬盘/
└── RecON_data/
    ├── calib_matrix.csv
    └── frames_transfs/
        ├── 001/
        │   ├── LH_Par_C_DtP.h5
        │   └── ...
        ├── 002/ ...
        └── 005/ ...
```

In [ ]:
DRIVE_DATA_DIR = '/content/drive/MyDrive/RecON_data'  # ← 若 Drive 中路径不同请修改

# 检查 Drive 中数据是否存在
assert os.path.exists(DRIVE_DATA_DIR), f'未找到 Drive 数据目录: {DRIVE_DATA_DIR}'
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'frames_transfs')), '缺少 frames_transfs 目录'
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'calib_matrix.csv')), '缺少 calib_matrix.csv'

# 用软链接将 Drive 数据映射到项目的 data/ 目录（无需复制，节省空间和时间）
local_data_dir = os.path.join(PROJECT_DIR, 'data')
os.makedirs(local_data_dir, exist_ok=True)

def make_symlink(src, dst):
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f'链接: {dst} → {src}')

make_symlink(
    os.path.join(DRIVE_DATA_DIR, 'frames_transfs'),
    os.path.join(local_data_dir, 'frames_transfs')
)
make_symlink(
    os.path.join(DRIVE_DATA_DIR, 'calib_matrix.csv'),
    os.path.join(local_data_dir, 'calib_matrix.csv')
)

# 验证数据可读
import h5py, glob
h5_files = glob.glob(os.path.join(local_data_dir, 'frames_transfs', '**', '*.h5'), recursive=True)
print(f'\n找到 {len(h5_files)} 个 h5 文件')
with h5py.File(h5_files[0], 'r') as f:
    print(f'示例文件: {h5_files[0]}')
    print(f'  键: {list(f.keys())}')
    print(f'  frames shape: {f["frames"].shape}')
    print(f'  tforms shape: {f["tforms"].shape}')

## 第五步：配置模型保存路径

将 `save/` 目录映射到 Drive，确保训练断点和结果在会话结束后不丢失。

In [ ]:
DRIVE_SAVE_DIR = '/content/drive/MyDrive/RecON_save'  # ← 可自定义
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

local_save_dir = os.path.join(PROJECT_DIR, 'save')
make_symlink(DRIVE_SAVE_DIR, local_save_dir)

print(f'检查点将保存至: {DRIVE_SAVE_DIR}')

## 第六步：运行训练

参数说明：
- `-m`：模型配置（`res/models/` 下的 JSON，不含扩展名）
- `-r`：运行超参数（`res/run/` 下的 JSON）
- `-d`：数据集配置（`res/datasets/` 下的 JSON）
- `-g`：GPU 编号（Colab 单卡用 `0`）

In [ ]:
%cd {PROJECT_DIR}

# 训练 Backbone
!python main.py \
    -m online_bk \
    -r hp_bk \
    -d TUS \
    -g 0

In [ ]:
# 从指定 epoch 继续训练（断点续训）
# RESUME_EPOCH = 50  # ← 修改为已保存的 epoch 号
# !python main.py -m online_bk -r hp_bk -d TUS -g 0  # 程序会自动读取最新检查点

In [ ]:
# 仅推理（指定 epoch）
TEST_EPOCH = 250  # ← 修改为目标 epoch
!python main.py \
    -m online_bk \
    -r hp_bk \
    -d TUS \
    -g 0 \
    -t {TEST_EPOCH}

## 附：快速环境检查（每次重连后执行）

Colab 实例重启后需重新挂载 Drive 并重建软链接，代码如下。

In [ ]:
# 重连后一键恢复（将第一步～第五步合并）
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR    = '/content/RecON'
DRIVE_DATA_DIR = '/content/drive/MyDrive/RecON_data'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/RecON_save'
GITHUB_REPO    = 'https://github.com/<YOUR_USERNAME>/RecON.git'  # ← 修改
BRANCH         = 'main'

# 克隆或更新代码
if not os.path.exists(PROJECT_DIR):
    os.system(f'git clone --branch {BRANCH} {GITHUB_REPO} {PROJECT_DIR}')

# 安装依赖
os.system('pip install timm pyvista h5py -q')

# 建立软链接
def make_symlink(src, dst):
    if os.path.lexists(dst): os.remove(dst)
    os.symlink(src, dst)

os.makedirs(os.path.join(PROJECT_DIR, 'data'), exist_ok=True)
make_symlink(os.path.join(DRIVE_DATA_DIR, 'frames_transfs'), os.path.join(PROJECT_DIR, 'data', 'frames_transfs'))
make_symlink(os.path.join(DRIVE_DATA_DIR, 'calib_matrix.csv'), os.path.join(PROJECT_DIR, 'data', 'calib_matrix.csv'))
make_symlink(DRIVE_SAVE_DIR, os.path.join(PROJECT_DIR, 'save'))

os.chdir(PROJECT_DIR)

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('环境就绪，可以运行 main.py')